# OSM Commercial Data Fetcher
**15-minute walk radius · OpenStreetMap via Overpass API**

This notebook downloads all commercial features (shops, offices, amenities, etc.) within a walking-distance radius of any coordinate and exports the result to a timestamped CSV ready for ML analysis.

---

### Team notes
- **Run cells top to bottom** — each section depends on the ones above.
- The only cell you normally need to edit is **Cell 3 — Parameters**. Set your origin coordinates there.
- Output CSVs are auto-named with an index and timestamp (`osm_commercial_001_2026-04-30_14-30-00.csv`) and saved in the same folder as this notebook.
- CSVs are excluded from git (see `.gitignore`). Commit the notebook, not the data files.
- If the Overpass API is slow or returns an error, the scraper will automatically retry on mirror servers.

---

### Prerequisites
Make sure you are running this notebook with the **`OSMnx Scraper (Python 3.11)`** kernel.  
To switch kernel in VS Code: `Ctrl+Shift+P` → *Notebook: Select Notebook Kernel* → pick `OSMnx Scraper (Python 3.11)`.  
If the kernel is missing, run `SETUP.md` instructions first.

## 1 · Imports

In [ ]:
import overpy
import requests
import pandas as pd
import math
import time
import glob
from datetime import datetime

print(f"overpy    {overpy.__version__}")
print(f"pandas    {pd.__version__}")
print(f"requests  {requests.__version__}")
print(f"Ready — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2 · OSM Tag Configuration

These constants define **what counts as commercial** in OpenStreetMap.

- `COMMERCIAL_TAGS` — OSM keys that identify commercial features. The order matters: the first matching key becomes the `primary_tag` for each record.
- `AMENITY_EXCLUDE` — amenity values that are infrastructure/civic, not commercial. Extend this list if you see noise in your results.
- `OVERPASS_ENDPOINTS` — mirror servers tried in order. The scraper uses `requests` with a proper `User-Agent` header to avoid being blocked by the API.

> **Only edit this cell if you want to change the scope of what gets collected.**

In [ ]:
# OSM keys that represent commercial land uses (priority order)
COMMERCIAL_TAGS = [
    "shop",
    "amenity",
    "office",
    "craft",
    "tourism",
    "leisure",
]

# Amenity values that are NOT commercial — filtered out before saving
AMENITY_EXCLUDE = {
    "parking", "bench", "waste_basket", "recycling",
    "post_box", "bicycle_parking", "drinking_water",
    "telephone", "toilets", "street_lamp", "fire_station",
    "school", "place_of_worship", "clock",
}

# Overpass API mirrors — tried in order if the primary server fails
# The scraper uses requests + User-Agent to avoid HTTP 406 blocks
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

print(f"Tracking {len(COMMERCIAL_TAGS)} tag types · {len(AMENITY_EXCLUDE)} amenity exclusions")

## 3 · Parameters  ← edit here

Set the origin point and walking radius for this scrape.

| Parameter | Description | Default |
|---|---|---|
| `LAT` / `LON` | Origin coordinate | Times Square, NYC |
| `WALK_MINUTES` | Radius expressed as walking time | 15 min |
| `WALK_SPEED_M_MIN` | Walking speed in meters/minute | 80 m/min ≈ 4.8 km/h |
| `OUTPUT_PATH` | Custom CSV filename, or `None` for auto | `None` (auto) |

> **Tip:** To find coordinates for any location, right-click on [Google Maps](https://maps.google.com) and copy the lat/lon.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────
LAT              = 40.7589    # Origin latitude
LON              = -73.9851   # Origin longitude
WALK_MINUTES     = 15.0       # Walking radius in minutes
WALK_SPEED_M_MIN = 80.0       # Walking speed (meters per minute)
OUTPUT_PATH      = None       # Set to "myfile.csv" to override auto-naming
# ────────────────────────────────────────────────────────

RADIUS_M = WALK_MINUTES * WALK_SPEED_M_MIN

# Auto-generate filename if not specified
if OUTPUT_PATH is None:
    existing     = glob.glob("osm_commercial_*.csv")
    next_index   = len(existing) + 1
    timestamp    = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    OUTPUT_PATH  = f"osm_commercial_{next_index:03d}_{timestamp}.csv"

print(f"Origin  : ({LAT}, {LON})")
print(f"Radius  : {RADIUS_M:.0f} m  ({WALK_MINUTES:.0f} min @ {WALK_SPEED_M_MIN:.0f} m/min)")
print(f"Output  : {OUTPUT_PATH}")

## 4 · Core Functions

These cells define the scraping pipeline. **Do not edit unless you are extending the tool.**

- `build_overpass_query` — assembles the Overpass QL query string
- `haversine` — computes straight-line distance in meters between two coordinates
- `_extract_record` — parses a single OSM node/way into a flat dictionary
- `_query_endpoint` — sends one HTTP POST with a `User-Agent` header, parses the response
- `fetch_osm_data` — tries each mirror in order, collects all nodes and ways
- `save_csv` — adds derived ML features (`has_name`, `distance_band`) and writes the CSV

In [ ]:
def build_overpass_query(lat: float, lon: float, radius: float) -> str:
    """Builds Overpass QL query for commercial nodes and ways within radius (meters)."""
    tag_filters = "\n    ".join(
        [f'node["{tag}"](around:{radius:.0f},{lat},{lon});' for tag in COMMERCIAL_TAGS]
        + [f'way["{tag}"](around:{radius:.0f},{lat},{lon});' for tag in COMMERCIAL_TAGS]
    )
    return f"""
[out:json][timeout:60];
(
    {tag_filters}
);
out center tags;
"""


def haversine(lat1, lon1, lat2, lon2) -> float:
    """Straight-line distance in meters between two lat/lon coordinates."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi       = math.radians(lat2 - lat1)
    dlambda    = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def _extract_record(osm_id, osm_type, lat, lon, tags, origin_lat, origin_lon) -> dict | None:
    """Parses a single OSM feature into a flat dict. Returns None if non-commercial."""
    primary_tag = primary_value = None
    for tag in COMMERCIAL_TAGS:
        if tag in tags:
            val = tags[tag]
            if tag == "amenity" and val in AMENITY_EXCLUDE:
                return None
            primary_tag, primary_value = tag, val
            break
    if not primary_tag:
        return None

    return {
        "osm_id"         : osm_id,
        "osm_type"       : osm_type,
        "lat"            : round(lat, 7),
        "lon"            : round(lon, 7),
        "distance_m"     : round(haversine(origin_lat, origin_lon, lat, lon), 1),
        "primary_tag"    : primary_tag,
        "primary_value"  : primary_value,
        "name"           : tags.get("name", ""),
        "brand"          : tags.get("brand", ""),
        "shop"           : tags.get("shop", ""),
        "amenity"        : tags.get("amenity", ""),
        "office"         : tags.get("office", ""),
        "craft"          : tags.get("craft", ""),
        "tourism"        : tags.get("tourism", ""),
        "leisure"        : tags.get("leisure", ""),
        "cuisine"        : tags.get("cuisine", ""),
        "opening_hours"  : tags.get("opening_hours", ""),
        "phone"          : tags.get("phone", ""),
        "website"        : tags.get("website", ""),
        "wheelchair"     : tags.get("wheelchair", ""),
        "outdoor_seating": tags.get("outdoor_seating", ""),
        "takeaway"       : tags.get("takeaway", ""),
        "delivery"       : tags.get("delivery", ""),
        "level"          : tags.get("level", ""),
        "addr_street"    : tags.get("addr:street", ""),
        "addr_housenumber": tags.get("addr:housenumber", ""),
        "addr_postcode"  : tags.get("addr:postcode", ""),
    }

In [ ]:
def _query_endpoint(endpoint: str, query: str) -> overpy.Result:
    """POSTs the Overpass query via requests with a User-Agent header to avoid HTTP 406 blocks."""
    response = requests.post(endpoint, data={"data": query}, headers=HEADERS, timeout=90)
    response.raise_for_status()
    return overpy.Overpass().parse_json(response.text)


def fetch_osm_data(lat: float, lon: float, radius: float) -> list[dict]:
    """Queries Overpass API with automatic failover across mirror servers."""
    query = build_overpass_query(lat, lon, radius)
    print(f"  → Querying Overpass API (radius: {radius:.0f} m)...")

    last_error = None
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            result = _query_endpoint(endpoint, query)
            print(f"  ✓ Success via {endpoint}")
            break
        except Exception as e:
            last_error = e
            print(f"  ✗ {endpoint} — {type(e).__name__}: {e}  (trying next mirror...)")
            time.sleep(3)
    else:
        raise RuntimeError(f"All Overpass API endpoints failed. Last error: {last_error}")

    records = []

    for node in result.nodes:
        rec = _extract_record(
            osm_id=node.id, osm_type="node",
            lat=float(node.lat), lon=float(node.lon),
            tags=node.tags, origin_lat=lat, origin_lon=lon,
        )
        if rec:
            records.append(rec)

    for way in result.ways:
        if hasattr(way, "center_lat") and way.center_lat:
            rec = _extract_record(
                osm_id=way.id, osm_type="way",
                lat=float(way.center_lat), lon=float(way.center_lon),
                tags=way.tags, origin_lat=lat, origin_lon=lon,
            )
            if rec:
                records.append(rec)

    print(f"  → {len(records)} commercial features found")
    return records


def save_csv(records: list[dict], output_path: str) -> pd.DataFrame:
    """Builds DataFrame, adds ML-ready derived columns, and writes CSV."""
    df = pd.DataFrame(records)

    # Binary flag: does the place have a name in OSM?
    df["has_name"] = df["name"].str.len().gt(0).astype(int)

    # Discretized distance band — useful as a categorical ML feature
    bins   = [0, 200, 400, 600, 800, 1000, 1200, 9999]
    labels = ["0-200m", "200-400m", "400-600m", "600-800m", "800-1000m", "1000-1200m", ">1200m"]
    df["distance_band"] = pd.cut(df["distance_m"], bins=bins, labels=labels)

    df.sort_values("distance_m", inplace=True)
    df.to_csv(output_path, index=False, encoding="utf-8")
    return df


print("Functions defined — ready to run.")

## 5 · Run the Scraper

This cell executes the full pipeline using the parameters set in **Cell 3**.  
The Overpass API query may take **10–30 seconds** depending on server load.

In [ ]:
records = fetch_osm_data(LAT, LON, RADIUS_M)

if not records:
    raise ValueError("No commercial features found in the specified area. Check your coordinates.")

df = save_csv(records, OUTPUT_PATH)

print(f"\n✓ {len(df)} records saved to '{OUTPUT_PATH}'")

## 6 · Explore Results

Quick inspection of the collected data. Use these cells to validate the output before passing it to your ML pipeline.

In [ ]:
# Preview the first rows
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.head(10)

In [ ]:
# Top 15 commercial categories by count
print("Top primary_value categories:")
df["primary_value"].value_counts().head(15)

In [ ]:
# Distribution by primary tag type (shop / amenity / office / etc.)
print("By primary_tag:")
df["primary_tag"].value_counts()

In [ ]:
# Distance band breakdown — how many features per walking zone?
print("By distance_band:")
df["distance_band"].value_counts().sort_index()

In [ ]:
# Column overview — useful when handing off to ML
df.dtypes